# z014 - Tendencias: promedio de los últimos 3 meses y comparación con el mes actual

**Idea:** además de lag y delta, ver si el mes actual está por encima o por debajo de su **promedio de 3 meses**.
(La pendiente de 3 puntos es igual a delta2 / 2, que ya tenemos, por eso se usa promedio y cociente.)

Para cada variable: `_media3` (promedio de t, t-1, t-2) y `_ratio3` (valor actual / media3).
Pasamos de 761 a ~1065 variables (dentro del rango 1000-1500 de la tutoría).
En 202104 la ventana tiene solo 2 meses (no hay 202102).

Trial 34, 538 árboles, 5 semillas, train 202104, valid 202106.

In [1]:
# --- 1. Librerías y parámetros comunes ---
import numpy as np
import pandas as pd
import lightgbm as lgb

GANANCIA_ACIERTO = 1_072_500
COSTO_ESTIMULO   = 27_500
MES_VALID = 202106
SEMILLAS  = [501419, 623353, 786349, 834569, 946331]
N_PARTICIONES = 50        # Shuffle Split del test: particiones al azar del 50% de 202106

# Trial 34 de z005 (el mejor en promedio de 5 semillas)
params = {
    "objective": "binary", "learning_rate": 0.005, "min_data_in_leaf": 0,
    "feature_fraction": 0.5, "feature_fraction_bynode": 0.2, "max_bin": 31, "verbose": -1,
    "num_leaves": 137, "min_sum_hessian_in_leaf": 9.755574,
}
ARBOLES = 538

In [2]:
# --- 2a. Cargar datos ---
df = pd.read_parquet("../competencia_01.parquet")
df["clase_binaria"] = (df["clase_ternaria"] == "BAJA+2").astype(int)
df = df.sort_values(["numero_de_cliente", "foto_mes"]).reset_index(drop=True)
no_usar = ["foto_mes", "clase_ternaria", "clase_binaria", "clase_binaria2"]

In [3]:
# --- 2b. (sin variables nuevas antes de los lags) ---
sin_lags = []

In [4]:
# --- 2c. Lags y deltas de orden 1 y 2 (sobre todas las columnas, incluidas las nuevas) ---
cols = [c for c in df.columns if c not in no_usar and c != "numero_de_cliente" and c not in sin_lags]
grupo = df.groupby("numero_de_cliente")[cols]
nuevas = []
for k in [1, 2]:
    lag = grupo.shift(k).astype("float32")
    delta = (df[cols] - lag).astype("float32")
    lag.columns   = [f"{c}_lag{k}" for c in cols]
    delta.columns = [f"{c}_delta{k}" for c in cols]
    nuevas += [lag, delta]
df = pd.concat([df] + nuevas, axis=1)
del nuevas, lag, delta, grupo

variables = [c for c in df.columns if c not in no_usar]
valid = df[df["foto_mes"] == MES_VALID]
ganancia_cliente = np.where(valid["clase_ternaria"].values == "BAJA+2", GANANCIA_ACIERTO, -COSTO_ESTIMULO)
print("Variables:", len(variables))

Variables: 761


In [5]:
# --- 2d. Tendencias: media de 3 meses y ratio ---
base_cols = [c for c in cols]
media3 = (df.groupby("numero_de_cliente")[base_cols].rolling(3, min_periods=1).mean()
            .reset_index(level=0, drop=True).sort_index().astype("float32"))
ratio3 = (df[base_cols] / media3.replace(0, np.nan)).astype("float32")
media3.columns = [f"{c}_media3" for c in base_cols]
ratio3.columns = [f"{c}_ratio3" for c in base_cols]
df = pd.concat([df, media3, ratio3], axis=1)
del media3, ratio3
vars_actual = variables
variables = [c for c in df.columns if c not in no_usar]
valid = df[df["foto_mes"] == MES_VALID]
print("Variables con tendencias:", len(variables))

Variables con tendencias: 1065


In [6]:
# --- 3. Funciones de evaluación ---
def metricas(prob, mascara=None):
    # Ganancia acumulada ordenando de mayor a menor probabilidad (opcional: solo una parte de los clientes)
    g = ganancia_cliente if mascara is None else ganancia_cliente[mascara]
    p = prob if mascara is None else prob[mascara]
    acum = np.cumsum(g[np.argsort(p)[::-1]])
    n = int(np.argmax(acum)) + 1
    return acum[n - 1] / 1e6, n, acum[min(11_999, len(acum) - 1)] / 1e6

def resumen(probs_por_version):
    """probs_por_version = {version: [prob_semilla1, ..., prob_semilla5]}"""
    # a) Por semilla: ganancia máxima, mejor N y top 12.000
    filas = []
    for v, lista in probs_por_version.items():
        for s, pr in zip(SEMILLAS, lista):
            g, n, g12 = metricas(pr)
            filas.append({"version": v, "semilla": s, "ganancia_max_M": round(g, 1), "mejor_n": n, "top12000_M": round(g12, 1)})
    por_semilla = pd.DataFrame(filas)
    tabla = por_semilla.groupby("version", sort=False)[["ganancia_max_M", "top12000_M", "mejor_n"]].agg(["mean", "std"]).round(1)

    # b) Shuffle Split del test: ensamble de las 5 semillas evaluado en 50 particiones del 50% de 202106
    rng = np.random.default_rng(SEMILLAS[0])
    ens = {v: np.mean(lista, axis=0) for v, lista in probs_por_version.items()}
    part = []
    for i in range(N_PARTICIONES):
        mascara = rng.random(len(ganancia_cliente)) < 0.5
        part.append({v: metricas(pr, mascara)[0] for v, pr in ens.items()})
    part = pd.DataFrame(part)
    tabla[("shuffle_split", "mean")] = part.mean().round(1)
    tabla[("shuffle_split", "std")] = part.std().round(1)
    tabla[("shuffle_split", "veces_mejor")] = part.idxmax(axis=1).value_counts().reindex(tabla.index).fillna(0).astype(int)
    tabla[("ensamble_5", "ganancia_max_M")] = [round(metricas(ens[v])[0], 1) for v in tabla.index]
    return por_semilla, tabla

def anotar(tabla, prefijo, meses_train, extra=""):
    ruta_res = "resultados_experimentos.csv"
    res = pd.read_csv(ruta_res, sep=";", decimal=",", encoding="utf-8-sig")
    res = res[~res["descripcion"].str.startswith(prefijo)]   # sin duplicar si se vuelve a correr
    for v, r in tabla.iterrows():
        nueva = {
            "fecha": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M"),
            "descripcion": (f"{prefijo} {v}: prom. {len(SEMILLAS)} semillas; shuffle split {r[('shuffle_split', 'mean')]} "
                            f"(mejor en {r[('shuffle_split', 'veces_mejor')]}/{N_PARTICIONES}){extra}"),
            "meses_train": str(meses_train), "mes_valid": MES_VALID, "semilla": str(SEMILLAS),
            "params": str(params), "mejor_n": int(r[("mejor_n", "mean")]),
            "ganancia_max_M": r[("ganancia_max_M", "mean")], "ganancia_top12000_M": r[("top12000_M", "mean")],
        }
        res = pd.concat([res, pd.DataFrame([nueva])], ignore_index=True)
    res["n_enviados"] = res["n_enviados"].astype("Int64")
    res.to_csv(ruta_res, sep=";", decimal=",", index=False, encoding="utf-8-sig")
    print("Anotado en", ruta_res)

In [7]:
# --- 4. Entrenar las dos versiones ---
train = df[df["foto_mes"] == 202104]
probs = {}
for version, vars_v in [("Actual", vars_actual), ("Con tendencias", variables)]:
    datos = lgb.Dataset(train[vars_v], label=train["clase_binaria"])
    modelos = [lgb.train({**params, "seed": s}, datos, num_boost_round=ARBOLES) for s in SEMILLAS]
    probs[version] = [m.predict(valid[vars_v]) for m in modelos]
    print(version, len(vars_v), "variables, listo")

imp = pd.Series(np.mean([m.feature_importance("gain") for m in modelos], axis=0), index=variables)
imp = imp / imp.sum() * 100
for t in ["_media3", "_ratio3"]:
    print(t, "importancia total:", round(imp[[v for v in variables if v.endswith(t)]].sum(), 1), "%")

Actual 761 variables, listo


Con tendencias 1065 variables, listo
_media3 importancia total: 24.1 %
_ratio3 importancia total: 18.0 %


In [8]:
# --- 5. Comparación ---
por_semilla, tabla = resumen(probs)
print(por_semilla.pivot(index="semilla", columns="version", values="ganancia_max_M"))
tabla

version  Actual  Con tendencias
semilla                        
501419    400.5           396.5
623353    397.2           399.2
786349    401.4           407.5
834569    397.2           407.2
946331    401.8           400.1


ganancia_max_M      top12000_M       mejor_n          \
                         mean  std       mean  std     mean     std   
version                                                               
Actual                  399.6  2.3      392.9  3.7  13212.0  1072.1   
Con tendencias          402.1  5.0      386.3  5.2  14289.8   699.3   

               shuffle_split                       ensamble_5  
                        mean   std veces_mejor ganancia_max_M  
version                                                        
Actual                 199.1  11.6          13          402.4  
Con tendencias         201.9  12.5          37          408.0

In [9]:
# --- 6. Anotar ---
anotar(tabla, "z014 tendencias", [202104])

Anotado en resultados_experimentos.csv


## Conclusiones (08/10)
- Con tendencias (~1065 variables): **402,1 ± 5,0 M** vs 399,6 → +2,5 M, dentro del ruido; Shuffle Split 37/50.
- En top 12.000 pierde (386,3 vs 392,9) y el mejor N sube a ~14.300. No es una mejora clara.